# 8 - Multi-Agent Research
## Subagent Delegation + Typed Handoff Contracts
24 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/8-multi-agent-research/multi_agent_research_workbook.ipynb)

One LLM trying to do everything produces mediocre results at every step. The multi-agent pattern assigns each step to a specialized agent, connected by typed handoffs.

This example builds a three-agent pipeline:
1. Supervisor - refines a vague topic into a focused research question
2. Researcher - extracts structured findings (ResearchFindings)
3. Writer - turns findings into an executive brief (WrittenBrief)

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | Why multiple agents beat one general-purpose agent |
| 2 | The handoff contract - typed schemas between agents |
| 3 | Build the Supervisor |
| 4 | Build the Researcher |
| 5 | Build the Writer |
| 6 | Run the full pipeline |
| star | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab
- OPENAI_API_KEY in .env or Colab Secrets

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

---

## Part 1 - Why Multiple Agents?

| Approach | Problem |
|----------|----------|
| One prompt, one call | Jack of all trades, master of none. |
| Chained agents, same prompt | No specialization. |
| Specialized agents, typed handoffs | Each agent has one job. Quality improves at every step. |

The researcher does not know what the writer will do with the findings. The writer does not know what the supervisor asked. Each agent receives exactly what it needs and returns exactly what the next agent expects.

---

## Part 2 - The Handoff Contract

The schema at each handoff is the contract:

```
User topic (string)
    |
    v
[SUPERVISOR]  --> refined_question (string)
    |
    v
[RESEARCHER]  --> ResearchFindings (typed Pydantic object)
    |
    v
[WRITER]  --> WrittenBrief (typed Pydantic object)
```

Swap out the researcher for a different implementation and the writer still works - as long as the new researcher returns a valid ResearchFindings.

---

## Part 3 - Build the Supervisor

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

SUPERVISOR_SYSTEM = SystemMessage(
    "You are a research supervisor. Turn vague topics into focused, answerable research questions.\n"
    "Example: 'AI in healthcare' -> 'What are the current clinical applications of AI in diagnostic imaging?'\n"
    "Return only the refined research question - one sentence."
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
supervisor = SUPERVISOR_SYSTEM | llm

TOPIC = "The impact of AI agents on software development workflows"
refined = supervisor.invoke(HumanMessage(content=f"Topic: {TOPIC}"))
refined_question = refined.content.strip()
print(f"Original topic:   {TOPIC}")
print(f"Refined question: {refined_question}")

---

## Part 4 - Build the Researcher

In [ ]:
from typing import List
from pydantic import BaseModel, Field


class ResearchFindings(BaseModel):
    topic: str
    key_facts: List[str] = Field(description="3-5 concrete facts")
    trends: List[str] = Field(description="2-3 observed trends")
    gaps: List[str] = Field(description="Questions this research could not answer")


RESEARCHER_SYSTEM = SystemMessage(
    "You are a research analyst. Produce structured findings on a given question.\n"
    "Be specific. Flag uncertain facts in gaps rather than stating them as fact.\n"
    "Do not invent statistics."
)

researcher = RESEARCHER_SYSTEM | llm.with_structured_output(ResearchFindings)
research = researcher.invoke(f"Research question: {refined_question}")

print(f"Topic: {research.topic}")
print(f"\nKey facts ({len(research.key_facts)}):")
for f in research.key_facts:
    print(f"  - {f}")
print(f"\nTrends ({len(research.trends)}):")
for t in research.trends:
    print(f"  - {t}")
print(f"\nGaps ({len(research.gaps)}):")
for g in research.gaps:
    print(f"  - {g}")
print(f"\nReturn type: {type(research).__name__}")

---

## Part 5 - Build the Writer

In [ ]:
class WrittenBrief(BaseModel):
    title: str
    executive_summary: str = Field(description="2-3 sentence summary for a busy executive")
    body: str = Field(description="400-600 word brief in markdown format")
    key_takeaways: List[str] = Field(description="3 bullet points the reader should remember")
    further_reading: List[str] = Field(description="2-3 topic areas worth exploring next")


WRITER_SYSTEM = SystemMessage(
    "You are a business writer producing executive-level research briefs.\n"
    "Lead with the most important insight. Use plain English. Markdown with ## headers.\n"
    "Synthesize and frame - do not just repeat the findings verbatim."
)

# Serialize the typed ResearchFindings into the writer prompt
writer_input = (
    f"Research question: {refined_question}\n\n"
    "Key facts:\n" + "\n".join(f"- {f}" for f in research.key_facts) + "\n\n"
    "Trends:\n" + "\n".join(f"- {t}" for t in research.trends) + "\n\n"
    "Gaps:\n" + "\n".join(f"- {g}" for g in research.gaps)
)

writer = WRITER_SYSTEM | llm.with_structured_output(WrittenBrief)
brief = writer.invoke(writer_input)

print(f"Title: {brief.title}")
print(f"\nExecutive Summary:\n{brief.executive_summary}")
print("\nKey Takeaways:")
for kt in brief.key_takeaways:
    print(f"  * {kt}")

---

## Part 6 - Full Pipeline

In [ ]:
def run_pipeline(topic: str) -> dict:
    "Supervisor -> Researcher -> Writer. Returns dict with refined_question, research, brief."
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    refined = (SUPERVISOR_SYSTEM | llm).invoke(HumanMessage(content=f"Topic: {topic}"))
    refined_q = refined.content.strip()

    research = (RESEARCHER_SYSTEM | llm.with_structured_output(ResearchFindings)).invoke(
        f"Research question: {refined_q}"
    )

    writer_input = (
        f"Research question: {refined_q}\n\n"
        "Key facts:\n" + "\n".join(f"- {f}" for f in research.key_facts) + "\n\n"
        "Trends:\n" + "\n".join(f"- {t}" for t in research.trends) + "\n\n"
        "Gaps:\n" + "\n".join(f"- {g}" for g in research.gaps)
    )
    brief = (WRITER_SYSTEM | llm.with_structured_output(WrittenBrief)).invoke(writer_input)

    return {"refined_question": refined_q, "research": research, "brief": brief}


result2 = run_pipeline("Climate tech investment trends")
print(f"Refined: {result2['refined_question']}")
print("\nBrief body (excerpt):")
print(result2['brief'].body[:600])

---

## Exercise 1 - Add a Fact-Checker Agent

Add a fourth agent between the Researcher and Writer.
The fact-checker receives ResearchFindings and returns a FactCheckResult
with verified_facts and questionable_claims.

The writer should only receive verified_facts.

In [ ]:
# Exercise 1 Starter
from pydantic import BaseModel, Field
from typing import List

class FactCheckResult(BaseModel):
    verified_facts: List[str] = Field(description="Facts that appear credible and well-supported")
    questionable_claims: List[str] = Field(description="Claims that seem overstated or unverifiable")

# TODO: write FACT_CHECKER_SYSTEM prompt
# TODO: fact_checker = FACT_CHECKER_SYSTEM | llm.with_structured_output(FactCheckResult)
# TODO: run it on research.key_facts + research.trends
# TODO: pass only fact_check.verified_facts to the writer

In [ ]:
# Exercise 1 Answer Key
FACT_CHECKER_SYSTEM = SystemMessage(
    "You are a fact-checker reviewing research findings for an executive brief.\n"
    "For each claim, assess whether it is:\n"
    "  verified_fact: concrete, specific, plausible\n"
    "  questionable_claim: vague, likely exaggerated, or unverifiable\n"
    "Be strict. Move borderline claims to questionable_claims."
)

fact_checker = FACT_CHECKER_SYSTEM | llm.with_structured_output(FactCheckResult)

all_claims = research.key_facts + research.trends
fact_check = fact_checker.invoke(
    "Claims to review:\n" + "\n".join(f"- {c}" for c in all_claims)
)

print(f"Verified facts ({len(fact_check.verified_facts)}):")
for f in fact_check.verified_facts:
    print(f"  ok  {f}")
print(f"\nQuestionable ({len(fact_check.questionable_claims)}):")
for q in fact_check.questionable_claims:
    print(f"  ?   {q}")

---

## Exercise 2 - Try Your Own Topic

Run the full pipeline on a topic of your choice.

Suggestions:
- The economics of remote work post-pandemic
- Geopolitical risks in global semiconductor supply chains
- Gene therapy breakthroughs in rare diseases

In [ ]:
MY_TOPIC = "The economics of remote work post-pandemic"

my_result = run_pipeline(MY_TOPIC)
print(f"Refined: {my_result['refined_question']}")
print(f"\nExecutive Summary:\n{my_result['brief'].executive_summary}")
print("\nKey Takeaways:")
for kt in my_result['brief'].key_takeaways:
    print(f"  * {kt}")

---

## What You Have Built

Here is what the full agent-use-cases series covered:

| # | Example | Pattern |
|---|---------|--------|
| 1 | Basic ReAct Agent | Reason -> Act -> Observe loop |
| 2 | Email Triage | Schema-constrained output |
| 3 | Invoice Extractor | Nested schemas + field validation |
| 4 | Lead Qualifier | Rubric in prompt + grounded reasoning |
| 5 | Support Ticket Router | Classification -> routing -> escalation gate |
| 6 | Resume Screener | Comparable structured output across many inputs |
| 7 | RFP Parser | Long-context extraction |
| 8 | Multi-Agent Research | Supervisor + subagent delegation + typed handoffs |

These patterns compose. A production system might combine a supervisor (8), structured extraction (3/7), and a routing layer (5) - all sharing typed schemas as the glue.

---
*Example 8 of 8 - agent-use-cases*